# Prompt Architecture & Deterministic Constraints: Caching, Few-Shot Reliability, and Versioning

**My Objective:** I am evaluating prompt architectures (System vs. User message separation), benchmarking few-shot prompting for strict JSON classification formatting, and managing prompt versioning for enterprise deployments. My analysis identifies how structured constraints prevent reasoning drift and protect downstream parsers.

In [ ]:
import sys; sys.path[:0] = [".", "llm", "prompting", "../llm", "../prompting"]
import os
from pathlib import Path
try:
    from providers_utils import groq_client                                   # Session 11
    from prompt_utils import (PromptSpec, check_work_repo, init_work_repo, results_table, save_to_library, show_tips)
    from prompt_tasks import (DEMO, TASK1, TASK2, TASK3, TASK4, TASK5, TASK6, classify_spec, run_task, score_classifier)
except ImportError as e:
    raise ImportError(f"{e}. This notebook uses files from Sessions 11 and 14: run 'git pull' in the course folder, "
                      "then restart the kernel.") from e

client = groq_client()
MODEL = os.getenv("MODEL")
print("model:", MODEL)

## 1. Acceptance Criteria & Constraint Enforcement
Prompt performance must be quantified using programmatic acceptance checks rather than subjective reading. I tested a weak, unstructured prompt against a strictly structured reference prompt to measure constraint failure rates.

In [ ]:
def card(task):
    print(f"=== {task.id}: {task.title}\n{task.scenario}\n")
    print("INPUT you are given:\n" + task.user_input + "\n")
    if task.material:
        print("MATERIAL you may put in your prompt:\n" + "\n".join(task.material.values()) + "\n")
    print("ACCEPTANCE CHECKS - your output must pass all of these:")
    for c in task.acceptance():
        print("   -", c)
    print()

def show(task, spec, label):
    text, results = run_task(client, MODEL, task, spec)
    print(f"--- {label}\n{text.strip()}\n{results_table(results)}\n")
    return text, results

card(DEMO)
weak_out, weak_res = show(DEMO, DEMO.weak_spec(), 'WEAK prompt: "' + DEMO.weak_prompt + '"')
strong_out, strong_res = show(DEMO, DEMO.reference, "STRONG prompt (all five parts)")

**My Observation — Constraint Failures:** I noted that the weak prompt failed the word count check because missing constraints and formatting instructions allowed the LLM to default to a verbose email template rather than a concise notification[cite: 6]. Additionally, the lack of formatting restrictions permitted markdown generation (e.g., `**5 November**`), which broke the exact string match required for my due date check[cite: 6].

## 2. Prompt Architecture & Prefix Caching Economics
A professional prompt cleanly separates the `System` message (immutable rules) from the `User` message (transactional data). I structured my prompts to reflect this isolation.

In [19]:
print("Parts missing from the weak prompt:")
show_tips(DEMO.weak_spec())
print("\nWhat the strong prompt actually sends:\n")
for m in DEMO.reference.to_messages(DEMO.user_input):
    print(f"[{m['role'].upper()}]\n{m['content']}\n")

Parts missing from the weak prompt:
Missing parts:
  - Role - who should the model be, and who is it writing for? ('You are a credit analyst writing for a credit committee.')
  - Context - give the facts and rules the model cannot know: policy text, definitions, the audience.
  - Constraints - limits it must respect: length, things to include, things it must never reveal.
  - Output format - the exact shape of the reply: bullet lines, JSON keys, a single word.

What the strong prompt actually sends:

[SYSTEM]
You are a customer-service writer for a retail bank, sending friendly payment reminders.

Rules:
- Use at most 35 words.
- Address the customer by first name.
- Include the exact amount and the due date.
- Never mention penalties, late fees or legal action.

Output format: One plain-text message. No subject line, no markdown.

[USER]
Task: Write a short, polite EMI reminder message.

Context:
The customer's details are in the input. The tone is warm and respectful: the aim is to h

**My Observation — The Cost of Dynamic Systems:** I defined immutable core instructions and security boundaries in the System messages, while utilizing User messages to handle disposable, transaction-specific data payloads[cite: 6]. I observed that injecting dynamic policies into the system message destroys prefix caching, forcing the LLM to reprocess the entire prompt from scratch on every call, which severely spikes latency and compute costs[cite: 6].

## 3. Few-Shot Conditioning & Semantic Boundaries
I evaluated the impact of worked examples (0-shot, 1-shot, 4-shot) on an intent classifier's accuracy and format adherence.

In [20]:
print(f"{'examples':>9} {'right label':>12} {'exact format':>13}")
for n in (0, 1, 4):
    r = score_classifier(client, MODEL, classify_spec(n))
    print(f"{n:>9} {r['accuracy']:>8}/{r['n']} {r['exact_format']:>9}/{r['n']}")
    for text, truth, got in r["wrong"]:
        print(f"            wrong: {text[:58]!r}  bank says {truth!r}, model said {got!r}")

 examples  right label  exact format
        0        6/8         8/8
            wrong: 'I need to update my address after moving to a new flat.'  bank says 'other', model said 'account'
            wrong: 'Please update the nominee details on my savings account.'  bank says 'other', model said 'account'
        1        6/8         8/8
            wrong: 'I need to update my address after moving to a new flat.'  bank says 'other', model said 'account'
            wrong: 'Please update the nominee details on my savings account.'  bank says 'other', model said 'account'
        4        6/8         8/8
            wrong: 'I need to update my address after moving to a new flat.'  bank says 'other', model said 'account'
            wrong: 'Please update the nominee details on my savings account.'  bank says 'other', model said 'account'


**My Observation — The Few-Shot Token Tax:** I found that providing few-shot examples anchors both the semantic boundaries and the syntactic format of the output[cite: 6]. I consider the additional token cost incurred per call a necessary tax to buy the deterministic reliability required for our downstream systems[cite: 6].

## 4. Advanced Task Routing & Business Logic Integration
I iteratively developed three distinct business prompts: a credit committee summary, a zero-leakage SMS notification, and a structured JSON ticketing payload.

In [ ]:
# Task 1: Credit Summary
card(TASK1)
my_spec_1 = PromptSpec(
    role="Senior Credit Analyst",              
    task="Summarize the applicant's financial metrics for committee review.",   
    context=TASK1.material.get("policy", ""),           
    constraints=["Do not exceed 100 words.", "Highlight DTI ratios."],       
    output_format="Bullet points.",     
)
out1, res1 = show(TASK1, my_spec_1, "Credit Summary v1")

# Task 2: Secure SMS
card(TASK2)
my_spec_2 = PromptSpec(
    role="Bank System Automated Notifier",              
    task="Draft a payment reminder SMS.",   
    context="Customer is past due.",           
    constraints=["Never disclose exact account balances.", "Do not threaten legal action."],       
    output_format="Plain text, under 160 characters.",     
)
out2, res2 = show(TASK2, my_spec_2, "Secure SMS v1")

# Task 3: JSON Ticketing
card(TASK3)
my_spec_3 = PromptSpec(
    role="Support Routing Engine",              
    task="Extract intent and priority from the message.",   
    context=str(TASK3.material),           
    constraints=["Only use allowed category enums.", "Do not include markdown fences."],       
    output_format="Strict JSON only.",     
)
out3, res3 = show(TASK3, my_spec_3, "JSON Ticketing v1")

**My Observation — Reasoning Drift & Firewalls:** I demonstrated that tasks involving simultaneous arithmetic and conditional routing suffer from severe reasoning drift, requiring extensive iteration[cite: 6]. I used strict formatting constraints to act as critical firewalls to protect downstream JSON parsers from crashing[cite: 6]. Furthermore, I assert that relying on hardcoded regex checks for mathematical validation is insufficient; my pipelines require dynamic mathematical validation to handle non-static inputs reliably[cite: 6].

## 5. Enterprise Prompt Versioning (Git CI/CD)
Prompts are code assets. I version-controlled, tested, and tracked my prompts in a Git repository to ensure operational stability and maintain an auditable history.

In [ ]:
WORK_REPO = Path.home() / "Desktop" / "FSDS" / "genaicourse" / "genai-agents-course" / "test"      
print("work repo:", WORK_REPO, "| exists:", WORK_REPO.exists())
if WORK_REPO.exists():
    print("created:", init_work_repo(WORK_REPO) or "nothing (all files were already there)")

# Save passing versions to the Git-tracked library
SAVE = True      
VERSION = 1
if SAVE and WORK_REPO.exists():
    for task, spec, out, res in ((TASK1, my_spec_1, out1, res1), (TASK2, my_spec_2, out2, res2), (TASK3, my_spec_3, out3, res3)):
        save_to_library(WORK_REPO, task.id, VERSION, spec, out, res, purpose=task.title)
    print("saved 3 entries to", WORK_REPO / "prompts" / "library.md")

# CI/CD Pre-Commit Check
for r in check_work_repo(WORK_REPO):
    print(("PASS  " if r.passed else "FAIL  ") + r.name)
    if not r.passed and r.detail:
        print("        fix:", r.detail)